In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.express as px
from itertools import product

from darpinstances.results import load_aggregate_stats_in_dir, load_occupancies_in_dir
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance.py:23: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## notes
- doesn't make sense to have dependent variable on x axis and independent variable on y axis -> it should be the other way around
- doesn't make sense to compare it across all instances (different config parameters) - choose one where all methods finish and where the instances are same except for on dependent variable that we want to compare it for
    - eg. occupancy based on delay - the instances should be same in: area, capacity, size (duration, sample) and if for all methods then the methods all need to finish
- could be configured: size (duration, sample), capacity, area, delay, method

# path setup

In [3]:
run_id = '3-run-14-4'

In [4]:
PATH = Path.cwd()
current_path = PATH
INSTANCE_PATH = PATH.parents[2] / "Instances"
RESULTS_PATH = PATH.parents[2] / "final-results" / run_id / "Results"

In [5]:
PATH = PATH.parents[2]
os.chdir(PATH)
IMG_PATH = PATH / "Ridesharing_DARP_instances/figures/bc-dominika"

In [6]:
Path.cwd()

PosixPath('/home/dominika/Desktop/deathOFbachelor')

In [ ]:
plt.rc('font', size=20)

In [ ]:
PATH

## results dataframe setup

In [27]:
def calculate_avg_across_option(value, option, data):
    match option:
        case 'method':
            # average cost across methods
            avg_df = data.groupby(
            ['method', 'area_short', 'duration_minutes', 'max_delay'],
            as_index=False)[value].mean()
        case 'sample':
            # average cost across capacities
            avg_df = data.groupby(
                ['sample', 'area_short', 'duration_minutes', 'max_delay'],
                as_index=False)[value].mean()
    avg_df['is_missing'] = avg_df[value].isna()
    return avg_df

In [20]:
def set_offset(option, data):
    order = sorted(data[option].unique())
    offsets = {
        opt: i - (len(order) - 1) / 2  # center around 0
        for i, opt in enumerate(order)
    }

    vals_to_axis = {}
    i = 1
    delays = sorted(data['max_delay'].unique(), reverse=True)
    durations = sorted(data['duration_minutes'].unique())
    for max_delay in delays:
        for duration in durations:
            vals_to_axis[(duration, max_delay)] = i
            i += 1
    return offsets, vals_to_axis

### average cost per request (old)

In [ ]:
dfih = df[df['method'] == 'ih']
fig = px.bar(
    dfih,
    x = 'area_short',
    y = 'cost_per_request',
    barmode='group',
    title = 'Average Cost per Request',
    facet_col='duration_minutes',
    facet_row='max_delay'
)

# shared axes titles
fig.for_each_yaxis(lambda y: y.update(title = ''))
fig.add_annotation(x=-0.06, y=0.5, text="travel time per request [s]", textangle=-90, xref="paper", yref="paper", showarrow=False)
fig.for_each_xaxis(lambda y: y.update(title = ''))

# faceting annotations
fig.add_annotation(x=0.5, y=1.15, text="instance length [min]",  xref="paper", yref="paper", showarrow=False)
fig.add_annotation(x=1.02, y=0.5, text="maximum delay [s]",  xref="paper", yref="paper", showarrow=False, textangle=90)

# faceting label editing
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
dfih['cost_per_request'].describe()

# Subsampling

### prepare occupancy dataframe

In [19]:
area = 'Manhattan'

In [15]:
sub_df = load_occupancies_in_dir(RESULTS_PATH / area)
sub_df['area'] = area

09:57:16 [INFO] Loading occupancy stats in /home/dominika/Desktop/deathOFbachelor/final-results/3-run-14-4/Results/Manhattan
09:57:16 [WARNING] No solution found in folder: /home/dominika/Desktop/deathOFbachelor/final-results/3-run-14-4/Results/Manhattan/start_18-00/duration_05_min/max_delay_10_min/sample_0.6/vga_chaining
09:57:16 [WARNING] No solution found in folder: /home/dominika/Desktop/deathOFbachelor/final-results/3-run-14-4/Results/Manhattan/start_18-00/duration_05_min/max_delay_10_min/sample_0.6/halns
09:57:16 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/3-run-14-4/Results/Manhattan/start_18-00/duration_05_min/max_delay_10_min/sample_0.6/ih/config.yaml-solution.json
09:57:16 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/3-run-14-4/Results/Manhattan/start_18-00/duration_05_min/max_delay_10_min/sample_0.6/ih/config.yaml-performance.json
09:57:16 [INFO] Loading experiment config from /home/dominika/Deskt

In [16]:
sub_df['cost_per_request'] = sub_df['cost_minutes'] * 60 / sub_df['req_count']
sub_df['area_short'] = sub_df['area'].map({
    'Manhattan': 'MH'
})
area_order = {
    'Manhattan': 3,
}
sub_df['area_order'] = sub_df['area'].map(area_order)
sub_df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)

### data analysis - cost

In [28]:
sample_filtered = sub_df[sub_df['cost_per_request'] > 0][['max_delay', 'method', 'area_short', 'cost_per_request', 'duration_minutes', 'sample']].drop_duplicates()

In [38]:
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = sample_filtered['method'].unique()
durations = sample_filtered['duration_minutes'].unique()
delays = sample_filtered['max_delay'].unique()
samples = sample_filtered['sample'].unique()

all_combinations = pd.DataFrame(
    list(product(methods, durations, delays, samples)),
    columns=['method', 'duration_minutes', 'max_delay', 'sample']
)

# Merge with the filtered data to find missing combinations
sample_complete = all_combinations.merge(sample_filtered, on=['method', 'duration_minutes', 'max_delay', 'sample'], how='left')

# # Add a column to mark missing values
# sample_complete['is_missing'] = sample_complete['cost_per_request'].isna()

# # Replace missing cost_per_request with 0 for plotting
# sample_complete['cost_per_request'] = sample_complete['cost_per_request'].fillna(0)

In [13]:
sorted_sample_df = sample_complete.sort_values(by=['sample', 'duration_minutes'])
fig = px.bar(
    sorted_sample_df,
    x='max_delay',
    y='cost_per_request',
    color='method',
    barmode='group',
    title='Average cost per request by method and area',
    facet_col='duration_minutes',
    facet_row='sample',
)

fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# unique_delays = sorted_sample_df['max_delay'].unique()
# fig.update_xaxes(tickmode='array', tickvals=unique_delays)

fig.show()

In [39]:
choose_fighter = 'method'
# choose_fighter = 'capacity'
avg_cost_df = calculate_avg_across_option('cost_per_request', choose_fighter, sample_complete)
offsets, vals_to_axis = set_offset(choose_fighter, avg_cost_df)

In [40]:
fig = px.bar(
    sample_complete,
    x='sample',
    y='cost_per_request',
    color='method',
    barmode='group',
    # title='Average cost per request by method and area',
    facet_row='max_delay',
    facet_col='duration_minutes',
)

# Add X annotations for missing method/area combos
# for _, row in avg_cost_df[avg_cost_df['is_missing']].iterrows():
#     axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

#     xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
#     yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

#     offset = offsets[row[choose_fighter]] * 0.3  # tweak this for spacing
#     fig.add_annotation(
#         x=row['area_short'],
#         y=0,
#         text="X",
#         xanchor='center',
#         yanchor='bottom',
#         showarrow=False,
#         font=dict(color='black', size=10),
#         xref=xref,
#         yref=yref,
#         xshift=offset * 40  # pixel offset for visual spacing
#     )
#     # 0.3, 50

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.03, y=0.5, text="Average cost per request [s]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.01, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.update_traces(textfont_size=7)
# fig.write_image(f"{IMG_PATH}/avg_cost_{choose_fighter}.png", width=1200, height=800)
fig.show()

## Upsampling
filling up higher capacities (eg. 6, 10) in Manhattan could have higher potential in Manhattan
- but load it to RCI cluster separately for each capacity? - duration X delay X capacity X sample = 4 x 4 x 3 x 7, where samples are 1.1, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0
- upsampling remains same but times should be regenerated (for all or only upsampled?) to make it more randomized (and conform to real-world demand) - from start to end time (use [generate_uniform_trip_times()](../darpinstances/instance_generation/demand_generation.py))